# Transfer Learning: Duck vs Chicken Classifier

Fine-tuning a pretrained **ResNet18** (trained on ImageNet) for binary image classification on a custom dataset of chicken and duck images.



**Pipeline:** upload dataset → flatten folder structure → clean → balance classes → train/test split → freeze ResNet18 backbone → replace final layer → train → evaluate.

## Setup

In [1]:
import os
import shutil
import random
import zipfile
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from sklearn.metrics import classification_report

torch.manual_seed(42)
random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## Step 1 — Upload the dataset zip

Running this cell opens a file picker. I select my `DUCK_CHICKEN_DATASET.zip` file from my computer. The zip will be extracted to the working directory.

In [2]:
from google.colab import files
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('.')
print(f"\nExtracted {zip_name}")
print("Top-level contents:", os.listdir('.'))

Saving DUCK_CHICKEN_DATASET.zip to DUCK_CHICKEN_DATASET.zip

Extracted DUCK_CHICKEN_DATASET.zip
Top-level contents: ['.config', 'DUCK_CHICKEN_DATASET.zip', 'DUCK_CHICKEN_DATASET', 'sample_data']


## Step 2 — Flatten the folder structure

The dataset is organized as `CHICKEN/{train,val,test}/` and `DUCK/{train,val,test}/`. PyTorch's `ImageFolder` expects a flat structure where each class is a single subfolder. We merge all splits into `data/chicken/` and `data/duck/`, prefixing filenames with the split name to avoid collisions.

In [3]:
SOURCE_ROOT = "DUCK_CHICKEN_DATASET"
DATA_DIR = "data"

if os.path.exists(DATA_DIR):
    shutil.rmtree(DATA_DIR)
os.makedirs(f"{DATA_DIR}/chicken", exist_ok=True)
os.makedirs(f"{DATA_DIR}/duck", exist_ok=True)

class_map = {"CHICKEN": "chicken", "DUCK": "duck"}
splits = ["train", "val", "test"]
image_exts = (".jpg", ".jpeg", ".png")

for src_cls, dst_cls in class_map.items():
    for split in splits:
        src_path = f"{SOURCE_ROOT}/{src_cls}/{split}"
        if not os.path.exists(src_path):
            continue
        for fname in os.listdir(src_path):
            if fname.lower().endswith(image_exts):
                shutil.copy(
                    os.path.join(src_path, fname),
                    os.path.join(DATA_DIR, dst_cls, f"{split}_{fname}")
                )

print(f"chicken: {len(os.listdir(f'{DATA_DIR}/chicken'))} images")
print(f"duck: {len(os.listdir(f'{DATA_DIR}/duck'))} images")

chicken: 499 images
duck: 1041 images


## Step 3 — Clean & balance the dataset

Remove unreadable files, then trim both classes to the same size so the model isn't biased toward the majority class.

In [4]:
def clean_images(folder):
    removed = 0
    for fname in os.listdir(folder):
        fpath = os.path.join(folder, fname)
        try:
            with Image.open(fpath) as img:
                img.verify()
            with Image.open(fpath) as img:
                img.convert("RGB")
        except Exception:
            os.remove(fpath)
            removed += 1
    return removed

for folder in ["chicken", "duck"]:
    path = f"{DATA_DIR}/{folder}"
    removed = clean_images(path)
    print(f"{folder}: {len(os.listdir(path))} valid (removed {removed} corrupt)")

min_count = min(len(os.listdir(f"{DATA_DIR}/chicken")), len(os.listdir(f"{DATA_DIR}/duck")))
for folder in ["chicken", "duck"]:
    files = sorted(os.listdir(f"{DATA_DIR}/{folder}"))
    random.Random(42).shuffle(files)
    for fname in files[min_count:]:
        os.remove(f"{DATA_DIR}/{folder}/{fname}")
print(f"\nBalanced both classes to {min_count} images each")

chicken: 499 valid (removed 0 corrupt)
duck: 1041 valid (removed 0 corrupt)

Balanced both classes to 499 images each


## Step 4 — Prepare DataLoaders

Resize to 224×224 and normalize using ImageNet mean/std (required since ResNet18 was pretrained on ImageNet). Then split 80/20 for train/test.

In [5]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataset = datasets.ImageFolder(DATA_DIR, transform=transform)
print(f"Classes: {dataset.classes}")
print(f"Total images: {len(dataset)}")

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_set, test_set = random_split(
    dataset, [train_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_set, batch_size=32, shuffle=True, num_workers=2)
test_loader = DataLoader(test_set, batch_size=32, shuffle=False, num_workers=2)
print(f"Train: {len(train_set)} | Test: {len(test_set)}")

Classes: ['chicken', 'duck']
Total images: 998
Train: 798 | Test: 200


## Step 5 — Load pretrained ResNet18

Load the ImageNet weights, **freeze** every parameter (so the backbone acts purely as a feature extractor), then replace the final fully-connected layer with a fresh one outputting 2 classes. Only this new layer will be trained.

In [6]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=1e-3)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 147MB/s]


## Step 6 — Train

Train only the final layer for 8 epochs.

In [7]:
EPOCHS = 8

for epoch in range(EPOCHS):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {running_loss/total:.4f} - Acc: {correct/total:.4f}")

Epoch 1/8 - Loss: 0.4967 - Acc: 0.7832
Epoch 2/8 - Loss: 0.2620 - Acc: 0.9223
Epoch 3/8 - Loss: 0.1960 - Acc: 0.9449
Epoch 4/8 - Loss: 0.1661 - Acc: 0.9486
Epoch 5/8 - Loss: 0.1707 - Acc: 0.9424
Epoch 6/8 - Loss: 0.1580 - Acc: 0.9411
Epoch 7/8 - Loss: 0.1444 - Acc: 0.9511
Epoch 8/8 - Loss: 0.1313 - Acc: 0.9549


## Step 7 — Evaluate

In [8]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(classification_report(all_labels, all_preds, target_names=dataset.classes))

              precision    recall  f1-score   support

     chicken       0.94      0.95      0.95       100
        duck       0.95      0.94      0.94       100

    accuracy                           0.94       200
   macro avg       0.95      0.94      0.94       200
weighted avg       0.95      0.94      0.94       200

